In [1]:
!pip install xgboost

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.1.2 -> 24.0
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
%pip install pyarrow

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.1.2 -> 24.0
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import glob
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split


In [2]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/20 05:08:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/20 05:08:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
/home/airflow/.local/lib/python3.7/site-packages/pyspark/context.py:317: FutureWarning: Python 3.7 support is deprecated in Spark 3.4.
  warnings.warn("Python 3.7 support is deprecated in Spark 3.4.", FutureWarning)


In [3]:
import pandas as pd

merged_tb = pd.read_parquet("/app/datamart/gold/training_dataset/merged_tb.parquet")

merged_tb.head()

,Customer_ID,snapshot_date,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,...,fe_12,fe_13,fe_14,fe_15,fe_16,fe_17,fe_18,fe_19,fe_20,label
0,cus_0x1037,2023-01-01,45.0,15989.085,1086.424,5.0,4.0,2.0,4.0,"[credit-builder loan, auto loan, mortgage loan]",...,105.0,-16.0,-81.0,-126.0,114.0,35.0,85.0,-73.0,76.0,0
1,cus_0x1069,2023-01-01,32.0,58637.340,4799.445,4.0,6.0,10.0,3.0,"[personal loan, auto loan]",...,190.0,-14.0,-96.0,200.0,35.0,130.0,94.0,111.0,75.0,0
2,cus_0x114a,2023-01-01,43.0,15305.460,1230.455,0.0,7.0,2.0,2.0,"[student loan, home equity loan]",...,203.0,26.0,86.0,171.0,125.0,-130.0,257.0,17.0,262.0,0
3,cus_0x1184,2023-01-01,49.0,19867.475,1396.623,3.0,5.0,11.0,3.0,"[student loan, mortgage loan, payday loan]",...,197.0,172.0,96.0,174.0,163.0,37.0,207.0,180.0,118.0,0
4,cus_0x1297,2023-01-01,46.0,57738.060,4881.505,9.0,8.0,30.0,9.0,"[payday loan, personal loan, mortgage loan, ho...",...,12.0,76.0,43.0,183.0,159.0,-26.0,104.0,118.0,184.0,1


In [4]:
merged_tb.columns

Index(['Customer_ID', 'snapshot_date', 'Age', 'Annual_Income',
       'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card',
       'Interest_Rate', 'Num_of_Loan', 'Type_of_Loan', 'Delay_from_due_date',
       'Num_of_Delayed_Payment', 'Changed_Credit_Limit',
       'Num_Credit_Inquiries', 'Outstanding_Debt', 'Credit_Utilization_Ratio',
       'Credit_History_Age', 'Total_EMI_per_month', 'Amount_invested_monthly',
       'Monthly_Balance', 'num_loan_types', 'debt_to_income', 'emi_to_salary',
       'investment_rate', 'balance_to_debt', 'inq_per_loan', 'fe_1', 'fe_2',
       'fe_3', 'fe_4', 'fe_5', 'fe_6', 'fe_7', 'fe_8', 'fe_9', 'fe_10',
       'fe_11', 'fe_12', 'fe_13', 'fe_14', 'fe_15', 'fe_16', 'fe_17', 'fe_18',
       'fe_19', 'fe_20', 'label'],
      dtype='object')

In [5]:
from sklearn.model_selection import train_test_split

training_set = merged_tb.dropna(subset=["label"]).copy()

drop_cols = [
    "Customer_ID",
    "snapshot_date",
    "Type_of_Loan",
    "label",
]

feature_cols = [
    c for c in training_set.columns
    if c not in drop_cols
]

X = training_set[feature_cols]
y = training_set["label"].astype(int)

# Keep numeric columns only
X = X.select_dtypes(include=["number"])

# Fill missing values using median
X = X.fillna(X.median())

# Optional checks
print("NaN count:", X.isna().sum().sum())
print("Shape:", X.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

NaN count: 0
Shape: (11973, 43)


In [7]:
# common imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

## Logistic regression

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    classification_report
)


log_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        C=1.0
    ))
])

log_model.fit(X_train, y_train)

print("Logistic Regression")

# Train results
print("=" * 50)
print("TRAIN RESULTS")
print("=" * 50)

log_train_pred = log_model.predict(X_train)
log_train_proba = log_model.predict_proba(X_train)[:, 1]

log_train_auc = roc_auc_score(y_train, log_train_proba)
log_train_acc = accuracy_score(y_train, log_train_pred)

print("ROC-AUC:", log_train_auc)
print("Accuracy:", log_train_acc)
print(classification_report(y_train, log_train_pred))

# Test results
print("=" * 50)
print("TEST RESULTS")
print("=" * 50)

log_test_pred = log_model.predict(X_test)
log_test_proba = log_model.predict_proba(X_test)[:, 1]

log_test_auc = roc_auc_score(y_test, log_test_proba)
log_test_acc = accuracy_score(y_test, log_test_pred)

print("ROC-AUC:", log_test_auc)
print("Accuracy:", log_test_acc)
print(classification_report(y_test, log_test_pred))

# Gap
print("Train-Test ROC-AUC Gap:", log_train_auc - log_test_auc)

Logistic Regression
TRAIN RESULTS
ROC-AUC: 0.7832516891174998
Accuracy: 0.7437878471497181
              precision    recall  f1-score   support

           0       0.86      0.77      0.81      6810
           1       0.54      0.69      0.61      2768

    accuracy                           0.74      9578
   macro avg       0.70      0.73      0.71      9578
weighted avg       0.77      0.74      0.75      9578

TEST RESULTS
ROC-AUC: 0.7800209762438947
Accuracy: 0.7448851774530272
              precision    recall  f1-score   support

           0       0.86      0.77      0.81      1703
           1       0.55      0.69      0.61       692

    accuracy                           0.74      2395
   macro avg       0.70      0.73      0.71      2395
weighted avg       0.77      0.74      0.75      2395

Train-Test ROC-AUC Gap: 0.0032307128736051016


In [14]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__penalty": ["l1", "l2"],
    "model__solver": ["liblinear"]
}

lr_grid = GridSearchCV(
    log_model,
    param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1
)

lr_grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('model',
                                        LogisticRegression(class_weight='balanced',
                                                           max_iter=1000))]),
             n_jobs=-1,
             param_grid={'model__C': [0.01, 0.1, 1, 10],
                         'model__penalty': ['l1', 'l2'],
                         'model__solver': ['liblinear']},
             scoring='roc_auc')

In [15]:
import pandas as pd

results = pd.DataFrame(lr_grid.cv_results_)
# Sort by rank to see the best models at the top
results.sort_values(by="rank_test_score", inplace=True)
results[["params", "mean_test_score", "std_test_score", "rank_test_score"]].head()


,params,mean_test_score,std_test_score,rank_test_score
2,"{'model__C': 0.1, 'model__penalty': 'l1', 'mod...",0.778094,0.007733,1
4,"{'model__C': 1, 'model__penalty': 'l1', 'model...",0.777770,0.007658,2
3,"{'model__C': 0.1, 'model__penalty': 'l2', 'mod...",0.777758,0.007672,3
6,"{'model__C': 10, 'model__penalty': 'l1', 'mode...",0.777723,0.007692,4
1,"{'model__C': 0.01, 'model__penalty': 'l2', 'mo...",0.777719,0.007665,5


In [16]:
print("Best Hyperparameters:", lr_grid.best_params_)
print("Best ROC-AUC Score:", lr_grid.best_score_)


Best Hyperparameters: {'model__C': 0.1, 'model__penalty': 'l1', 'model__solver': 'liblinear'}
Best ROC-AUC Score: 0.7780942637505001


## Random forest

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42
)

rf_model.fit(X_train, y_train)

# Train results
rf_train_pred = rf_model.predict(X_train)
rf_train_proba = rf_model.predict_proba(X_train)[:, 1]

rf_train_auc = roc_auc_score(y_train, rf_train_proba)
rf_train_acc = accuracy_score(y_train, rf_train_pred)

print("=" * 50)
print("RANDOM FOREST - TRAIN RESULTS")
print("=" * 50)
print("ROC-AUC:", rf_train_auc)
print("Accuracy:", rf_train_acc)
print(classification_report(y_train, rf_train_pred))


# Test results
rf_test_pred = rf_model.predict(X_test)
rf_test_proba = rf_model.predict_proba(X_test)[:, 1]

rf_test_auc = roc_auc_score(y_test, rf_test_proba)
rf_test_acc = accuracy_score(y_test, rf_test_pred)

print("=" * 50)
print("RANDOM FOREST - TEST RESULTS")
print("=" * 50)
print("ROC-AUC:", rf_test_auc)
print("Accuracy:", rf_test_acc)
print(classification_report(y_test, rf_test_pred))

print("=" * 50)
print("Train-Test ROC-AUC Gap:", rf_train_auc - rf_test_auc)

RANDOM FOREST - TRAIN RESULTS
ROC-AUC: 0.9204199133372377
Accuracy: 0.7989141783253288
              precision    recall  f1-score   support

           0       0.89      0.82      0.85      6810
           1       0.63      0.74      0.68      2768

    accuracy                           0.80      9578
   macro avg       0.76      0.78      0.77      9578
weighted avg       0.81      0.80      0.80      9578

RANDOM FOREST - TEST RESULTS
ROC-AUC: 0.8089710779006107
Accuracy: 0.7691022964509394
              precision    recall  f1-score   support

           0       0.86      0.80      0.83      1703
           1       0.59      0.68      0.63       692

    accuracy                           0.77      2395
   macro avg       0.72      0.74      0.73      2395
weighted avg       0.78      0.77      0.77      2395

Train-Test ROC-AUC Gap: 0.11144883543662698


In [18]:

rf_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "max_features": ["sqrt", "log2"],
    "model__criterion": ["gini", "entropy"]
}  


In [25]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)

rf_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "max_features": ["sqrt", "log2"],
    "criterion": ["gini", "entropy"]
}

grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=rf_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 108 candidates, totalling 540 fits


GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [5, 10, 20],
                         'max_features': ['sqrt', 'log2'],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 200, 300]},
             scoring='roc_auc', verbose=1)

In [26]:
import pandas as pd

print("Best parameters:")
print(grid_search.best_params_)

print("Best CV AUC:")
print(grid_search.best_score_)

rf_results = pd.DataFrame(grid_search.cv_results_)

rf_results = rf_results.sort_values(
    "mean_test_score",
    ascending=False
)

rf_results[
    [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "param_n_estimators",
        "param_max_depth",
        "param_min_samples_split",
        "param_max_features",
        "param_criterion"
    ]
].head(10)

Best parameters:
{'criterion': 'entropy', 'max_depth': 20, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 300}
Best CV AUC:
0.8135609466485768


,rank_test_score,mean_test_score,std_test_score,param_n_estimators,param_max_depth,param_min_samples_split,param_max_features,param_criterion
92,1,0.813561,0.008389,300,20,2,sqrt,entropy
74,2,0.813032,0.008985,300,10,2,sqrt,entropy
95,3,0.812965,0.009545,300,20,5,sqrt,entropy
98,4,0.812965,0.009279,300,20,10,sqrt,entropy
91,5,0.812906,0.008613,200,20,2,sqrt,entropy
107,6,0.812710,0.008570,300,20,10,log2,entropy
94,7,0.812628,0.008680,200,20,5,sqrt,entropy
106,8,0.812627,0.008262,200,20,10,log2,entropy
77,9,0.812230,0.009522,300,10,5,sqrt,entropy
104,10,0.812055,0.009460,300,20,5,log2,entropy


## XGBoost

In [10]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

# class imbalance weight
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(X_train, y_train)

# Train results
xgb_train_pred = xgb_model.predict(X_train)
xgb_train_proba = xgb_model.predict_proba(X_train)[:, 1]

xgb_train_auc = roc_auc_score(y_train, xgb_train_proba)
xgb_train_acc = accuracy_score(y_train, xgb_train_pred)

print("=" * 50)
print("XGBOOST - TRAIN RESULTS")
print("=" * 50)
print("ROC-AUC:", xgb_train_auc)
print("Accuracy:", xgb_train_acc)
print(classification_report(y_train, xgb_train_pred))

# Test results
xgb_test_pred = xgb_model.predict(X_test)
xgb_test_proba = xgb_model.predict_proba(X_test)[:, 1]

xgb_test_auc = roc_auc_score(y_test, xgb_test_proba)
xgb_test_acc = accuracy_score(y_test, xgb_test_pred)

print("=" * 50)
print("XGBOOST - TEST RESULTS")
print("=" * 50)
print("ROC-AUC:", xgb_test_auc)
print("Accuracy:", xgb_test_acc)
print(classification_report(y_test, xgb_test_pred))

print("Train-Test ROC-AUC Gap:", xgb_train_auc - xgb_test_auc)

XGBOOST - TRAIN RESULTS
ROC-AUC: 0.9385879794674612
Accuracy: 0.8586343704322406
              precision    recall  f1-score   support

           0       0.94      0.86      0.90      6810
           1       0.71      0.86      0.78      2768

    accuracy                           0.86      9578
   macro avg       0.82      0.86      0.84      9578
weighted avg       0.87      0.86      0.86      9578

XGBOOST - TEST RESULTS
ROC-AUC: 0.8208270681795811
Accuracy: 0.7795407098121085
              precision    recall  f1-score   support

           0       0.87      0.82      0.84      1703
           1       0.60      0.69      0.64       692

    accuracy                           0.78      2395
   macro avg       0.74      0.75      0.74      2395
weighted avg       0.79      0.78      0.78      2395

Train-Test ROC-AUC Gap: 0.11776091128788013


In [16]:
# xgb_grid = {
#     "n_estimators": [100, 200, 300],
#     "max_depth": [3, 5, 7],
#     "learning_rate": [0.01, 0.05, 0.1],
#     "subsample": [0.8, 1.0]
# }

In [17]:
# RandomizedSearchCV

## Compare all 3

In [11]:
results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Train ROC-AUC": log_train_auc,
        "Test ROC-AUC": log_test_auc,
        "Train Accuracy": log_train_acc,
        "Test Accuracy": log_test_acc,
    },
    {
        "Model": "Random Forest",
        "Train ROC-AUC": rf_train_auc,
        "Test ROC-AUC": rf_test_auc,
        "Train Accuracy": rf_train_acc,
        "Test Accuracy": rf_test_acc,
    },
    {
        "Model": "XGBoost",
        "Train ROC-AUC": xgb_train_auc,
        "Test ROC-AUC": xgb_test_auc,
        "Train Accuracy": xgb_train_acc,
        "Test Accuracy": xgb_test_acc,
    }
])

results["Gap"] = results["Train ROC-AUC"] - results["Test ROC-AUC"]

results = results.sort_values(
    by="Test ROC-AUC",
    ascending=False
).reset_index(drop=True)

print(results)

                 Model  Train ROC-AUC  Test ROC-AUC  Train Accuracy  \
0              XGBoost       0.938588      0.820827        0.858634   
1        Random Forest       0.920420      0.808971        0.798914   
2  Logistic Regression       0.783252      0.780021        0.743788   

   Test Accuracy       Gap  
0       0.779541  0.117761  
1       0.769102  0.111449  
2       0.744885  0.003231  


In [12]:
import os
import pickle
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# 1. Set model training date


from datetime import datetime

model_train_date_str = datetime.today().strftime("%Y-%m-%d")


# 2. Collect trained models

trained_models = {
    "logistic_regression": log_model,
    "random_forest": rf_model,
    "xgboost": xgb_model
}


# 3. Evaluate all models

model_results = []

for model_name, model in trained_models.items():

    train_proba = model.predict_proba(X_train)[:, 1]
    test_proba = model.predict_proba(X_test)[:, 1]

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_auc = roc_auc_score(y_train, train_proba)
    test_auc = roc_auc_score(y_test, test_proba)

    model_results.append({
        "model_name": model_name,
        "auc_train": train_auc,
        "auc_test": test_auc,
        "gini_train": round(2 * train_auc - 1, 3),
        "gini_test": round(2 * test_auc - 1, 3),
        "accuracy_test": accuracy_score(y_test, test_pred),
        "precision_test": precision_score(y_test, test_pred),
        "recall_test": recall_score(y_test, test_pred),
        "f1_test": f1_score(y_test, test_pred),
        "train_rows": X_train.shape[0],
        "test_rows": X_test.shape[0],
        "train_default_rate": round(y_train.mean(), 2),
        "test_default_rate": round(y_test.mean(), 2)
    })

model_metrics = pd.DataFrame(model_results).sort_values(
    "auc_test",
    ascending=False
)

display(model_metrics)


# 4. Select champion model

champion_model_name = model_metrics.iloc[0]["model_name"]
champion_model = trained_models[champion_model_name]

print("Champion model:", champion_model_name)


# 5. Create model bank folder

model_bank_directory = "model_bank"

os.makedirs(model_bank_directory, exist_ok=True)


# 6. Save all 3 model artefacts

saved_artefacts = []

for model_name, model in trained_models.items():

    row = model_metrics[
        model_metrics["model_name"] == model_name
    ].iloc[0]

    model_version = (
        "credit_model_"
        + model_name
        + "_"
        + model_train_date_str.replace("-", "_")
    )

    model_artefact = {
        "model": model,
        "model_name": model_name,
        "model_version": model_version,
        "is_champion": model_name == champion_model_name,
        "feature_cols": feature_cols,
        "data_dates": {
            "model_train_date": model_train_date_str
        },
        "data_stats": {
            "X_train_rows": X_train.shape[0],
            "X_test_rows": X_test.shape[0],
            "y_train_default_rate": round(y_train.mean(), 2),
            "y_test_default_rate": round(y_test.mean(), 2)
        },
        "results": {
            "auc_train": row["auc_train"],
            "auc_test": row["auc_test"],
            "gini_train": row["gini_train"],
            "gini_test": row["gini_test"],
            "accuracy_test": row["accuracy_test"],
            "precision_test": row["precision_test"],
            "recall_test": row["recall_test"],
            "f1_test": row["f1_test"]
        },
        "hp_params": model.get_params()
    }

    file_path = os.path.join(
        model_bank_directory,
        model_version + ".pkl"
    )

    with open(file_path, "wb") as file:
        pickle.dump(model_artefact, file)

    saved_artefacts.append({
        "model_name": model_name,
        "model_version": model_version,
        "is_champion": model_name == champion_model_name,
        "auc_test": row["auc_test"],
        "file_path": file_path
    })

    print(f"Saved {model_name} to {file_path}")


# 7. Save champion model

champion_version = (
    "credit_model_"
    + champion_model_name
    + "_"
    + model_train_date_str.replace("-", "_")
)

champion_source_path = os.path.join(
    model_bank_directory,
    champion_version + ".pkl"
)

champion_target_path = os.path.join(
    model_bank_directory,
    "champion_model.pkl"
)

with open(champion_source_path, "rb") as file:
    champion_artefact = pickle.load(file)

with open(champion_target_path, "wb") as file:
    pickle.dump(champion_artefact, file)

print("Champion model saved to:", champion_target_path)


# 8. Save registry and metrics

model_registry = pd.DataFrame(saved_artefacts)

model_registry.to_csv(
    os.path.join(model_bank_directory, "model_registry.csv"),
    index=False
)

model_metrics.to_csv(
    os.path.join(model_bank_directory, "model_metrics.csv"),
    index=False
)

display(model_registry)


# 9. Test load champion pickle

with open(champion_target_path, "rb") as file:
    loaded_champion_artefact = pickle.load(file)

loaded_model = loaded_champion_artefact["model"]
loaded_feature_cols = loaded_champion_artefact["feature_cols"]

loaded_test_proba = loaded_model.predict_proba(
    X_test[loaded_feature_cols]
)[:, 1]

loaded_test_auc = roc_auc_score(
    y_test,
    loaded_test_proba
)

print("Loaded champion model:", loaded_champion_artefact["model_name"])
print("Loaded champion version:", loaded_champion_artefact["model_version"])
print("Loaded champion test AUC:", loaded_test_auc)
print("Model loaded successfully!")

,model_name,auc_train,auc_test,gini_train,gini_test,accuracy_test,precision_test,recall_test,f1_test,train_rows,test_rows,train_default_rate,test_default_rate
2,xgboost,0.938588,0.820827,0.877,0.642,0.779541,0.603275,0.692197,0.644684,9578,2395,0.29,0.29
1,random_forest,0.920420,0.808971,0.841,0.618,0.769102,0.586121,0.683526,0.631087,9578,2395,0.29,0.29
0,logistic_regression,0.783252,0.780021,0.567,0.560,0.744885,0.546605,0.686416,0.608584,9578,2395,0.29,0.29


Champion model: xgboost
Saved logistic_regression to model_bank/credit_model_logistic_regression_2026_06_20.pkl
Saved random_forest to model_bank/credit_model_random_forest_2026_06_20.pkl
Saved xgboost to model_bank/credit_model_xgboost_2026_06_20.pkl
Champion model saved to: model_bank/champion_model.pkl


,model_name,model_version,is_champion,auc_test,file_path
0,logistic_regression,credit_model_logistic_regression_2026_06_20,False,0.780021,model_bank/credit_model_logistic_regression_20...
1,random_forest,credit_model_random_forest_2026_06_20,False,0.808971,model_bank/credit_model_random_forest_2026_06_...
2,xgboost,credit_model_xgboost_2026_06_20,True,0.820827,model_bank/credit_model_xgboost_2026_06_20.pkl


Loaded champion model: xgboost
Loaded champion version: credit_model_xgboost_2026_06_20
Loaded champion test AUC: 0.8208270681795811
Model loaded successfully!
